# Análise de Crédito – CredAIR
Investigação dos fatores relacionados à inadimplência e desenvolvimento de um modelo de Regressão Logística para apoiar a identificação de propostas de maior risco.

## 1. Carregamento dos dados
Bibliotecas usadas na análise e leitura do arquivo `credito.csv`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/credito.csv")
df.head()

,idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
0,52,MS,Ensino Superior,2911.82,14,690,5934.19,60,Viagem,1,0
1,49,RR,Ensino Médio,3176.43,5,676,6791.38,24,Outros,2,0
2,48,AL,Ensino Superior,3429.40,23,558,14666.78,24,Outros,5,1
3,59,GO,Ensino Fundamental,10096.70,2,739,19053.24,24,Reforma,0,0
4,37,RJ,Ensino Fundamental,3033.25,24,797,6171.31,6,Educação,0,1


In [5]:
df = pd.read_csv("../data/credito.csv")

### Estrutura do dataset
Verificação da quantidade de registros, dos tipos das colunas e de valores ausentes.

In [6]:
print(df.shape)
df.info()

(5000, 11)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   idade               5000 non-null   int64  
 1   estado              5000 non-null   str    
 2   escolaridade        5000 non-null   str    
 3   renda_mensal        5000 non-null   float64
 4   tempo_emprego_anos  5000 non-null   int64  
 5   score_credito       5000 non-null   int64  
 6   valor_solicitado    5000 non-null   float64
 7   prazo_meses         5000 non-null   int64  
 8   finalidade          5000 non-null   str    
 9   dividas_ativas      5000 non-null   int64  
 10  inadimplente        5000 non-null   int64  
dtypes: float64(2), int64(6), str(3)
memory usage: 429.8 KB


O dataset tem 5.000 propostas e 11 colunas. Nenhuma coluna tem valores nulos e os tipos estão corretos. 
Isso não garante que os dados estejam limpos: ausentes podem estar representados por valores como 0 ou -1, 
o que será verificado a seguir.

### Estatísticas das variáveis numéricas
Análise de mínimos e máximos para identificar valores impossíveis ou fora da faixa esperada.

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
idade,5000.0,47.767400,31.646099,18.00,32.00,46.00,61.0000,750.00
renda_mensal,5000.0,5228.633582,3968.478198,100.00,2697.32,4146.30,6458.8775,44814.19
tempo_emprego_anos,5000.0,12.662200,7.517539,1.00,6.00,13.00,19.0000,74.00
score_credito,5000.0,652.481200,84.817076,335.00,594.00,653.00,710.0000,850.00
valor_solicitado,5000.0,14264.657378,8268.261465,1721.48,8467.17,12365.63,17764.8850,50000.00
prazo_meses,5000.0,29.922000,15.482515,6.00,18.00,24.00,36.0000,60.00
dividas_ativas,5000.0,1.900400,1.864719,0.00,0.00,1.00,3.0000,7.00
inadimplente,5000.0,0.344000,0.475089,0.00,0.00,0.00,1.0000,1.00


### Categorias das variáveis de texto
Verificação das categorias existentes e de possíveis grafias diferentes para o mesmo valor.

In [8]:
for col in ["estado", "escolaridade", "finalidade"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

--- estado ---
estado
DF        205
SC        197
MG        197
GO        195
PB        195
         ... 
  PB        1
  AM        1
 MG         1
 SE         1
  RR        1
Name: count, Length: 85, dtype: int64

--- escolaridade ---
escolaridade
Ensino Médio          2061
Ensino Superior       1938
Pós-graduação          590
Ensino Fundamental     411
Name: count, dtype: int64

--- finalidade ---
finalidade
Veículo     1128
Reforma     1117
Outros       941
Educação     709
Saúde        611
Viagem       494
Name: count, dtype: int64



In [9]:
print(df["estado"].nunique())
print(sorted(df["estado"].unique()))

85
['  AC ', '  AL ', '  AL  ', '  AM ', '  CE ', '  DF ', '  GO ', '  MG ', '  MS ', '  PA ', '  PB ', '  PB  ', '  PE  ', '  PI ', '  RO  ', '  RR ', '  RR  ', '  RS ', '  RS  ', '  SE ', '  SE  ', '  SP ', '  SP  ', ' AC  ', ' AL ', ' AM ', ' AM  ', ' AP  ', ' BA ', ' CE ', ' DF ', ' ES ', ' GO ', ' GO  ', ' MA ', ' MG ', ' MG  ', ' MS ', ' MT  ', ' PA ', ' PB ', ' PB  ', ' PE ', ' PE  ', ' PI ', ' PI  ', ' RJ ', ' RJ  ', ' RN ', ' RN  ', ' RO ', ' RR ', ' RR  ', ' SC ', ' SC  ', ' SE ', ' SE  ', ' TO ', 'AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']


A coluna `estado` apresenta 85 valores distintos, embora existam apenas 27 UFs. 
Há siglas com espaços extras (ex.: " MG"), que serão padronizadas na etapa de limpeza.

### Investigação dos valores suspeitos
Quantificação dos problemas indicados pelas estatísticas descritivas.

In [10]:
print("Idade > 100:", (df["idade"] > 100).sum())
print(df.loc[df["idade"] > 100, "idade"].value_counts())
print()
print("Renda igual a 100:", (df["renda_mensal"] == 100).sum())
print("Valor solicitado igual a 50.000:", (df["valor_solicitado"] == 50000).sum())
print("Tempo de emprego incompatível com a idade:", (df["tempo_emprego_anos"] > df["idade"] - 14).sum())
print()
print("Prazos existentes:", sorted(df["prazo_meses"].unique()))
print("Linhas duplicadas:", df.duplicated().sum())

Idade > 100: 15
idade
660    1
680    1
710    1
430    1
300    1
360    1
440    1
280    1
500    1
340    1
210    1
540    1
700    1
750    1
670    1
Name: count, dtype: int64

Renda igual a 100: 1
Valor solicitado igual a 50.000: 32
Tempo de emprego incompatível com a idade: 696

Prazos existentes: [np.int64(6), np.int64(12), np.int64(18), np.int64(24), np.int64(36), np.int64(48), np.int64(60)]
Linhas duplicadas: 19


### Resumo do diagnóstico

- 15 idades acima de 100, todas múltiplas de 10 (provável zero a mais na digitação);
- 696 registros com tempo de emprego incompatível com a idade;
- 85 valores distintos em `estado` por causa de espaços extras;
- 19 linhas duplicadas;
- 32 propostas no valor máximo de R\$ 50.000, provavelmente o teto de crédito (não é erro);
- 1 renda de R\$ 100, caso isolado.

Esses pontos serão tratados na etapa de limpeza.